# Core SWE — Principal Engineer Tracker

A study companion for the breadth of software engineering knowledge expected at principal/staff level:
OOP · Design Patterns · System Design · OS · DBMS · Networking · Concurrency · Git/Testing/Debugging

**How to use**

- Check off items as you gain confidence.
- Every code cell has `# Time:` and `# Space:` annotations — add your own when you practice.
- Each section ends with a practice cell. Write your own implementation *before* re-reading the intro code.
- For patterns and design topics, sketch the structure on paper first, then code it.

**Recommended routine**

- 1 section/week is sustainable alongside a full-time role.
- After each section, pick one real production problem at work and apply the pattern.
- Revisit the DSA notebook alongside System Design — many distributed systems problems map to graph/DP thinking.

---
# 1. Object-Oriented Programming

### Study checklist
- [ ] Classes, objects, and instance vs class attributes
- [ ] Encapsulation with properties
- [ ] Inheritance and the MRO
- [ ] Polymorphism and duck typing
- [ ] Abstract base classes (ABCs)
- [ ] Composition vs inheritance
- [ ] Magic / dunder methods

### Notes
Write design decisions, gotchas, and edge cases here.

### Beginner-friendly intro
OOP models the world as objects that hold both state (attributes) and behaviour (methods). The four pillars — encapsulation, inheritance, polymorphism, abstraction — are tools for managing complexity. At principal level the key question is: *which pillar applies here, and is it earning its weight?*

### Classes, objects, instance vs class attributes

**Approach:** Define what belongs to every instance vs what is shared across all instances. Instance attributes go in `__init__`; class attributes are declared at class scope. Mutable class-level containers (lists, dicts) are traps — they're effectively globals.

**Trade-offs:** Class attributes save memory when thousands of instances share a constant, but any mutable class attribute causes cross-instance bleed. Prefer `None`-default instance attributes over mutable class-level defaults.

**When to use it:** Class attributes for constants (`MAX_RETRIES = 3`), registries, and counters shared across all instances. Avoid class-level mutable containers (lists, dicts) as pseudo-defaults — every instance shares the same object unless you explicitly create a new one in `__init__`.

**Learn more:** [Python Data Model](https://docs.python.org/3/reference/datamodel.html)


In [ ]:
class BankAccount:
    INTEREST_RATE = 0.05   # class attribute: shared constant across all instances

    def __init__(self, owner: str, balance: float = 0.0):
        self.owner    = owner      # instance attribute: unique per object
        self._balance = balance    # underscore = internal by convention, not truly private

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError(f"Deposit must be positive, got {amount}")
        self._balance += amount

    def __repr__(self) -> str:
        return f"BankAccount(owner={self.owner!r}, balance={self._balance:.2f})"

acct = BankAccount("Alice", 100)
acct.deposit(50)
print(acct)
print(f"Rate (shared): {BankAccount.INTEREST_RATE}")
# Time:  O(1) for every operation — single arithmetic op or string format
# Space: O(1) per instance — fixed number of attributes regardless of balance size

### Encapsulation with properties

**Approach:** Use `@property` to expose a clean public interface while keeping validation inside the class. The caller writes `t.celsius = 100`, not `t.set_celsius(100)` — same syntax as a plain attribute, but guarded. Computed read-only properties derive from stored state without duplicating it.

**Trade-offs:** Properties add a method-call overhead on every access (tiny but nonzero). For hot loops over millions of objects, plain attributes or `__slots__` may be preferable. For most domain objects the clarity payoff outweighs the cost.

**When to use it:** Input validation (temperature, money), computed read-only views (fahrenheit, percentage), and lazy-loaded attributes. Avoid properties for simple data-holder objects with no validation or computation — a plain attribute or a `dataclass` field is clearer and faster.

**Learn more:** [Python Data Model](https://docs.python.org/3/reference/datamodel.html)


In [ ]:
class Temperature:
    def __init__(self, celsius: float):
        self.celsius = celsius        # route through setter from the start

    @property
    def celsius(self) -> float:
        return self._celsius

    @celsius.setter
    def celsius(self, value: float) -> None:
        if value < -273.15:
            raise ValueError(f"Below absolute zero: {value}")
        self._celsius = value

    @property
    def fahrenheit(self) -> float:    # computed, read-only — no setter needed
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
print(f"{t.celsius}°C = {t.fahrenheit}°F")
t.celsius = 100
print(f"{t.celsius}°C = {t.fahrenheit}°F")
try:
    t.celsius = -300
except ValueError as e:
    print(f"Rejected: {e}")
# Time:  O(1) per get/set — arithmetic only
# Space: O(1) — one stored value; fahrenheit is computed on demand

### Inheritance and the Method Resolution Order

**Approach:** Model 'is-a' relationships. Prefer narrow, shallow hierarchies — each level should add one clear invariant. Use `super()` for cooperative calls so mix-in chains resolve correctly. Read `ClassName.__mro__` when in doubt about which method will be called.

**Trade-offs:** Inheritance couples child tightly to parent internals (the fragile base class problem). Changing a parent method signature can silently break all children. Mix-ins are the safer multi-inheritance idiom — each adds one isolated behaviour.

**When to use it:** Framework lifecycle hooks, custom exceptions, and iterator/context-manager protocols where a genuine 'is-a' relationship holds. Avoid deep inheritance chains (more than 2-3 levels) or inheriting purely to reuse a few methods — that's what composition and mix-ins are for.

**Learn more:** [Python Data Model](https://docs.python.org/3/reference/datamodel.html)


In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name

    def speak(self) -> str:
        raise NotImplementedError(f"{type(self).__name__} must implement speak()")

class LogMixin:
    # Mix-in: adds logging without touching the Animal hierarchy
    def speak(self) -> str:
        result = super().speak()        # cooperative — works correctly in MRO chain
        print(f"[LOG] {self.name} said: {result}")
        return result

class Dog(Animal):
    def speak(self) -> str:
        return "Woof"

class LoggingDog(LogMixin, Dog):        # MRO: LoggingDog → LogMixin → Dog → Animal
    pass

fido = LoggingDog("Fido")
fido.speak()
print("MRO:", [c.__name__ for c in LoggingDog.__mro__])
# Time:  O(d) for method resolution where d = MRO depth (constant in practice)
# Space: O(d) for the __mro__ tuple stored once per class definition

### Polymorphism and duck typing

**Approach:** Write functions that call an interface (a set of method names), not a concrete type. Any object implementing that interface can be passed in — no explicit inheritance required. Use `isinstance()` only when you genuinely need to branch on type.

**Trade-offs:** Duck typing reduces boilerplate and coupling but shifts errors to runtime. Type hints + `mypy` recover static safety without requiring formal inheritance. ABCs provide a middle ground: formal interface + duck-typing flexibility.

**When to use it:** Plugin systems, rendering pipelines, serializers, and payment providers where callers only need a known set of methods. Avoid pure duck typing on public APIs consumed by external teams — an explicit ABC (next item) documents the contract and fails fast on a missing method.

**Learn more:** [Python Data Model](https://docs.python.org/3/reference/datamodel.html)


In [ ]:
import math

class Circle:
    def __init__(self, r: float): self.r = r
    def area(self) -> float:      return math.pi * self.r ** 2
    def perimeter(self) -> float: return 2 * math.pi * self.r

class Rectangle:
    def __init__(self, w: float, h: float): self.w, self.h = w, h
    def area(self) -> float:      return self.w * self.h
    def perimeter(self) -> float: return 2 * (self.w + self.h)

class Triangle:
    def __init__(self, a, b, c): self.sides = (a, b, c)
    def area(self) -> float:
        s = sum(self.sides) / 2
        return math.sqrt(s * (s-self.sides[0]) * (s-self.sides[1]) * (s-self.sides[2]))
    def perimeter(self) -> float: return sum(self.sides)

def print_stats(shape) -> None:        # accepts ANY object with .area()/.perimeter()
    print(f"{type(shape).__name__:<12} area={shape.area():>8.2f}  perim={shape.perimeter():>8.2f}")

for s in [Circle(5), Rectangle(4, 6), Triangle(3, 4, 5)]:
    print_stats(s)
# Time:  O(1) per shape — fixed-formula geometry
# Space: O(1) per instance — 1–3 stored floats

### Abstract base classes (ABCs)

**Approach:** Declare an interface contract with `ABC` + `@abstractmethod`. The interpreter enforces that concrete subclasses implement every abstract method before instantiation — fail fast at class-definition time. Add concrete methods on the ABC for shared behaviour every implementer would otherwise repeat.

**Trade-offs:** ABCs add a hard dependency in the hierarchy; pure duck typing is more flexible. Choose ABCs when you want the interpreter to catch missing implementations early, not at call time. Good for public plugin APIs where external authors will implement the interface.

**When to use it:** Declaring plugin contracts, I/O adapters (Serializer, Storage, Notifier), and protocols for external implementers. Avoid ABCs for internal-only code with a single implementation — the extra formality buys nothing until a second, independently-authored implementation actually exists.

**Learn more:** [Python Data Model](https://docs.python.org/3/reference/datamodel.html)


In [ ]:
from abc import ABC, abstractmethod

class Serializer(ABC):
    @abstractmethod
    def serialize(self, data: dict) -> str:
        pass  # Convert a dict to a wire-format string

    @abstractmethod
    def deserialize(self, raw: str) -> dict:
        pass  # Parse a wire-format string back to a dict

    def roundtrip(self, data: dict) -> dict:    # concrete method reusing abstract interface
        return self.deserialize(self.serialize(data))

class JsonSerializer(Serializer):
    def serialize(self, data):
        import json; return json.dumps(data, sort_keys=True)
    def deserialize(self, raw):
        import json; return json.loads(raw)

class CsvSerializer(Serializer):
    def serialize(self, data):
        return ",".join(f"{k}={v}" for k, v in sorted(data.items()))
    def deserialize(self, raw):
        return dict(pair.split("=") for pair in raw.split(","))

for ser in [JsonSerializer(), CsvSerializer()]:
    payload = {"name": "Alice", "score": "95"}
    ok = ser.roundtrip(payload) == payload
    print(f"{type(ser).__name__}: roundtrip_ok={ok}  wire={ser.serialize(payload)!r}")
# Time:  O(n) for serialize/deserialize where n = number of dict entries
# Space: O(n) for the string or dict produced

### Composition over inheritance

**Approach:** Model 'has-a' before 'is-a'. Inject dependencies through the constructor so each component is testable in isolation. Interfaces (ABCs or duck typing) define the contract; concrete classes satisfy it. The consumer calls the injected interface, never the concrete type directly.

**Trade-offs:** Composition requires wiring at construction time (more explicit) but avoids the fragile-base-class problem. Inheritance is appropriate for genuine 'is-a' relationships and when the subtype must pass `isinstance` checks. Default: try composition first.

**When to use it:** Logging backends, storage adapters, payment gateways, and notification channels — anywhere the concrete implementation should be swappable without touching the consumer. Avoid composition when the relationship is genuinely hierarchical and the subtype must pass `isinstance` checks elsewhere in the codebase; that's what inheritance is for.

**Learn more:** [Python Data Model](https://docs.python.org/3/reference/datamodel.html)


In [ ]:
class FileLogger:
    def log(self, msg: str) -> None: print(f"[FILE]  {msg}")

class CloudLogger:
    def log(self, msg: str) -> None: print(f"[CLOUD] {msg}")

class MultiLogger:
    # Compose multiple loggers (Open/Closed: extend without modifying)
    def __init__(self, *loggers):
        self._loggers = loggers

    def log(self, msg: str) -> None:
        for logger in self._loggers:
            logger.log(msg)

class OrderService:
    def __init__(self, logger):      # accepts anything with a .log() method
        self._logger = logger

    def place_order(self, order_id: str) -> None:
        self._logger.log(f"Order {order_id} placed")

# Swap logger without touching OrderService
svc1 = OrderService(MultiLogger(FileLogger(), CloudLogger()))
svc1.place_order("ORD-42")

svc2 = OrderService(CloudLogger())   # different logger, same service code
svc2.place_order("ORD-43")
# Time:  O(k) per log call where k = number of composed loggers
# Space: O(k) for the logger tuple in MultiLogger

### Magic / dunder methods

**Approach:** Implement dunders to integrate custom objects with Python syntax and built-ins. Always define `__repr__`. Define `__eq__` when value equality matters, and pair it with `__hash__` so the object works in sets and as dict keys. Add arithmetic dunders (`__add__`, `__rmul__`) for natural operator syntax.

**Trade-offs:** Dunders make objects feel native but can surprise readers if semantics are non-obvious. `__getattr__` and `__setattr__` are powerful but easy to cause infinite recursion with. Prefer explicit methods for non-obvious behaviour; reserve dunders for obvious Python protocols.

**When to use it:** Numeric value types, context managers (`__enter__`/`__exit__`), and custom containers (`__len__`, `__getitem__`). Avoid overriding `__getattr__`/`__setattr__` unless you specifically need dynamic attribute behavior — they're the most common source of infinite-recursion bugs in this category, and a plain method is safer for anything non-obvious.

**Learn more:** [Python Data Model](https://docs.python.org/3/reference/datamodel.html)


In [ ]:
class Money:
    def __init__(self, amount: float, currency: str = "USD"):
        self._amount   = round(amount, 2)
        self._currency = currency

    def __repr__(self) -> str:
        return f"Money({self._amount:.2f}, {self._currency!r})"

    def __add__(self, other: "Money") -> "Money":
        if self._currency != other._currency:
            raise ValueError(f"Cannot add {self._currency} and {other._currency}")
        return Money(self._amount + other._amount, self._currency)

    def __mul__(self, factor: float) -> "Money":
        return Money(self._amount * factor, self._currency)

    def __rmul__(self, factor: float) -> "Money":   # supports: 3 * price
        return self.__mul__(factor)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Money): return NotImplemented
        return self._amount == other._amount and self._currency == other._currency

    def __hash__(self) -> int:   # required whenever __eq__ is defined
        return hash((self._amount, self._currency))

price = Money(9.99)
tax   = Money(0.80)
print(price + tax)          # Money(10.79, 'USD')
print(3 * price)            # Money(29.97, 'USD')
print(price == Money(9.99)) # True
prices = {price, Money(5.00)}  # works as set element — __hash__ defined
print(f"Set size: {len(prices)}")
# Time:  O(1) for all arithmetic — fixed-precision float operations
# Space: O(1) per Money instance — two fields

### OOP practice cell

In [ ]:
# Design a TaskManager.
# Requirements:
#   - Tasks have: id, title, status (todo/in-progress/done), priority (1-5)
#   - TaskManager: add(), update_status(), list_by_priority(), search(keyword)
#   - Use @property for status with a whitelist validator
#   - Make Task __repr__ useful; make tasks sortable by priority with __lt__
#
# Problem: implement Task and TaskManager
# Approach:
# Time Complexity: list_by_priority, search
# Space Complexity:
# Edge Cases: duplicate titles, invalid status, empty manager, priority out of range

---
# 2. Design Patterns

### Study checklist
- [ ] Singleton — one instance, global access point
- [ ] Factory — decouple object creation from use
- [ ] Builder — construct complex objects step-by-step
- [ ] Observer — event-driven decoupling
- [ ] Strategy — swap algorithms at runtime
- [ ] Decorator — add behaviour without subclassing
- [ ] Adapter — bridge incompatible interfaces

### Notes
Write design decisions, gotchas, and edge cases here.

### Beginner-friendly intro
Design patterns are repeatable solutions to recurring design problems. They are vocabulary, not templates to copy verbatim. At principal level: recognise which pattern fits, know its trade-offs, and know when *not* to use it (over-engineering is a real cost).

#### 2.1 Singleton — one instance, global access point

**Approach:** Override `__new__` (not just `__init__`, which runs every time regardless) to return the same cached instance on every call. Guard the check-and-create step with a lock and re-check inside it (double-checked locking) so two threads racing to create the first instance can't both slip past the initial `None` check and construct two separate objects.

**When to use it:** Config/settings objects, connection-pool managers, loggers. Avoid for anything that needs to be tested in isolation — singletons make dependency injection hard.

**Trade-offs:** Global access is convenient, but it's really global mutable state wearing a class's clothing — any code anywhere can read or mutate it, which is exactly the kind of hidden coupling dependency injection is designed to avoid. Tests must explicitly reset `_instance` to `None` between runs, or state leaks across test cases in ways that are hard to trace back to this pattern specifically.

**Learn more:** [Refactoring.Guru: Design Patterns](https://refactoring.guru/design-patterns)


In [ ]:
import threading

class AppConfig:
    """Thread-safe singleton configuration store."""
    _instance = None
    _lock = threading.Lock()

    def __new__(cls, **kwargs):
        if cls._instance is None:
            with cls._lock:          # double-checked locking
                if cls._instance is None:
                    obj = super().__new__(cls)
                    obj._data = {}
                    cls._instance = obj
        return cls._instance

    def set(self, key, value): self._data[key] = value
    def get(self, key, default=None): return self._data.get(key, default)
    def __repr__(self): return f"AppConfig(id={id(self)}, keys={list(self._data)})"

# Time: O(1) amortised (lock only on first creation)
# Space: O(k) where k = number of config keys

cfg1 = AppConfig()
cfg2 = AppConfig()
cfg1.set("env", "prod")
print(cfg2.get("env"))        # prod — same object
print(cfg1 is cfg2)           # True

#### 2.2 Factory — decouple object creation from use

**Approach:** A registry dict maps string keys to classes; `create()` looks up the key and instantiates the class it finds. This isn't just shorter than an `if/elif` chain — it's Open/Closed: adding a new notifier type means registering one new entry, never touching the existing dispatch logic, so old, tested code paths can't regress from a new feature.

**When to use it:** Plugin systems, notification channels, codec selection — anywhere the concrete type is chosen at runtime from config or user input.

**Trade-offs:** The registry buys extensibility without modification, but it also means the concrete class actually instantiated for a given key is only knowable by reading the registry at runtime — a debugger stepping through `create()` sees a generic lookup, not the specific type, which costs a little traceability for the extensibility gain.

**Learn more:** [Refactoring.Guru: Design Patterns](https://refactoring.guru/design-patterns)


In [ ]:
class EmailNotifier:
    def send(self, msg): print(f"[EMAIL] {msg}")

class SMSNotifier:
    def send(self, msg): print(f"[SMS] {msg}")

class PushNotifier:
    def send(self, msg): print(f"[PUSH] {msg}")

class NotifierFactory:
    _registry = {
        "email": EmailNotifier,
        "sms":   SMSNotifier,
        "push":  PushNotifier,
    }

    @classmethod
    def register(cls, key, klass):
        cls._registry[key] = klass

    @classmethod
    def create(cls, channel: str):
        klass = cls._registry.get(channel)
        if klass is None:
            raise ValueError(f"Unknown channel: {channel!r}. "
                             f"Available: {list(cls._registry)}")
        return klass()

# Time: O(1) lookup
# Space: O(n) registry where n = registered types

for ch in ("email", "sms", "push"):
    NotifierFactory.create(ch).send("Hello")

#### 2.3 Builder — construct complex objects step-by-step

**Approach:** Each setter mutates the builder's own state and returns `self`, which is what makes the fluent chain (`.header(...).body(...).timeout(...)`) possible — without returning `self`, each call would need to be a separate statement. Validation is deferred to `build()` specifically because a partially-configured builder is expected to exist transiently; only the final, fully-configured object needs to satisfy the invariants (like 'POST requires a body').

**When to use it:** HTTP clients, query builders, test-data factories — objects with many optional parameters where a constructor would need 10+ args.

**Trade-offs:** The fluent call site reads almost like a spec, but you're now maintaining two classes (the builder and the immutable product) instead of one, and the builder's own state can be left invalid mid-chain — the design only pays off once the target object has enough optional parameters that a single constructor call would otherwise need 5+ keyword arguments.

**Learn more:** [Refactoring.Guru: Design Patterns](https://refactoring.guru/design-patterns)


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Optional

@dataclass(frozen=True)
class HttpRequest:
    method:  str
    url:     str
    headers: Dict[str, str]
    body:    Optional[str]
    timeout: int

class HttpRequestBuilder:
    def __init__(self, method: str, url: str):
        self._method  = method.upper()
        self._url     = url
        self._headers: Dict[str, str] = {}
        self._body:    Optional[str]  = None
        self._timeout: int            = 30

    def header(self, key: str, value: str) -> "HttpRequestBuilder":
        self._headers[key] = value
        return self

    def body(self, payload: str) -> "HttpRequestBuilder":
        self._body = payload
        return self

    def timeout(self, seconds: int) -> "HttpRequestBuilder":
        self._timeout = seconds
        return self

    def build(self) -> HttpRequest:
        if self._method in ("POST", "PUT", "PATCH") and self._body is None:
            raise ValueError(f"{self._method} request requires a body")
        return HttpRequest(self._method, self._url,
                           dict(self._headers), self._body, self._timeout)

# Time: O(1) per setter, O(h) for build where h = header count
# Space: O(h + len(body))

req = (HttpRequestBuilder("POST", "https://api.example.com/orders")
       .header("Content-Type", "application/json")
       .header("Authorization", "Bearer tok123")
       .body('{"item": "widget", "qty": 3}')
       .timeout(10)
       .build())
print(req)

#### 2.4 Observer — event-driven decoupling

**Approach:** An `EventBus` holds a dict mapping each event name to a list of subscribed callbacks; publishers never know who (if anyone) is listening, and subscribers never know who publishes — both sides only depend on the event name and payload shape, which is what actually decouples them, not the bus object itself.

**When to use it:** UI frameworks, domain events (OrderPlaced → trigger email + inventory update), plugin hooks.

**Trade-offs:** Decoupling publishers from subscribers means a stack trace at `emit()` time won't show you which subscribers will run or in what order — that information only exists in the `_listeners` dict at runtime, so debugging a multi-subscriber event chain often means adding temporary logging inside each callback rather than reading a call stack.

**Learn more:** [Refactoring.Guru: Design Patterns](https://refactoring.guru/design-patterns)


In [ ]:
from collections import defaultdict
from typing import Callable, Any

class EventBus:
    def __init__(self):
        self._listeners: dict[str, list[Callable]] = defaultdict(list)

    def on(self, event: str, callback: Callable) -> None:
        self._listeners[event].append(callback)

    def off(self, event: str, callback: Callable) -> None:
        self._listeners[event] = [cb for cb in self._listeners[event]
                                   if cb is not callback]

    def emit(self, event: str, **payload: Any) -> None:
        for cb in list(self._listeners[event]):   # copy — safe mid-loop removal
            cb(**payload)

# Time: O(s) per emit where s = subscriber count
# Space: O(e * s) where e = distinct events

bus = EventBus()

def send_confirmation(order_id, total, **_):
    print(f"[Email] Order {order_id} confirmed — ${total:.2f}")

def update_inventory(order_id, items, **_):
    print(f"[Inventory] Reserving {items} for order {order_id}")

bus.on("order.placed", send_confirmation)
bus.on("order.placed", update_inventory)
bus.emit("order.placed", order_id="ORD-99", total=42.50, items=["widget"])

#### 2.5 Strategy — swap algorithms at runtime

**Approach:** Accept a callable (or any object implementing `__call__`) at construction time and store it; every call to the context's own method just delegates to that stored strategy. This replaces a conditional (`if algorithm == 'x': ... elif ...`) with polymorphic dispatch — the context's code never needs to change when a new strategy is added, it just needs a new callable passed in.

**When to use it:** Sorting comparators, compression codecs, payment processors, A/B feature flags.

**Trade-offs:** Removing the conditional from the context pushes the decision of which strategy to use out to the caller — that's a net win for the context's testability (each strategy can be tested standalone), but it means the context alone can no longer answer 'what will this do', since the answer depends entirely on which strategy was injected.

**Learn more:** [Refactoring.Guru: Design Patterns](https://refactoring.guru/design-patterns)


In [ ]:
from typing import Callable, List

class Sorter:
    def __init__(self, strategy: Callable[[List], List]):
        self._strategy = strategy

    def sort(self, data: List) -> List:
        return self._strategy(list(data))   # non-destructive copy

# --- concrete strategies ---
def insertion_sort(arr):
    # Time: O(n²)  Space: O(1)
    a = arr[:]
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0 and a[j] > key:
            a[j+1] = a[j]
            j -= 1
        a[j+1] = key
    return a

def counting_sort_non_neg_int(arr):
    # Time: O(n+k)  Space: O(k)  where k = max value
    if not arr: return arr
    k = max(arr)
    counts = [0] * (k + 1)
    for x in arr: counts[x] += 1
    return [x for x, c in enumerate(counts) for _ in range(c)]

data = [5, 2, 8, 1, 9, 3]
print(Sorter(sorted).sort(data))               # built-in timsort
print(Sorter(insertion_sort).sort(data))       # O(n²) strategy
print(Sorter(counting_sort_non_neg_int).sort(data))  # O(n+k) strategy

#### 2.6 Decorator — add behaviour without subclassing

**Approach:** A wrapper function accepts the target callable, defines an inner function that runs code before and/or after calling it, and returns that inner function in its place. `functools.wraps` is essential, not cosmetic — without it, the wrapped function loses its original `__name__` and docstring, which breaks introspection, debugging tools, and anything relying on `repr(func)`.

**When to use it:** Logging, timing, caching, retry logic, auth checks — cross-cutting concerns.

**Trade-offs:** Each decorator stays focused on one cross-cutting concern (timing, caching, retries), but stacking several (`@timed @memoize def fib`) means a stack trace on failure shows wrapper frames from every decorator in the chain, not just the original function — worth it for the separation of concerns, but it does make tracebacks noisier to read.

**Learn more:** [Refactoring.Guru: Design Patterns](https://refactoring.guru/design-patterns)


In [ ]:
import time, functools
from typing import Callable

def timed(func: Callable) -> Callable:
    """Log call duration."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - t0
        print(f"[timed] {func.__name__} took {elapsed*1000:.2f} ms")
        return result
    return wrapper

def memoize(func: Callable) -> Callable:
    """Simple unbounded memoization."""
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    # expose cache for inspection / testing
    wrapper.cache = cache
    return wrapper

# Time: O(1) overhead per call (hash lookup)
# Space: O(n) where n = distinct arg tuples seen

@timed
@memoize
def fib(n: int) -> int:
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print(fib(35))
print(f"Cache size: {len(fib.cache)}")

#### 2.7 Adapter — bridge incompatible interfaces

**Approach:** Define the interface your application actually wants to depend on, then write a thin adapter class that implements that interface by translating each call into the legacy SDK's actual method names and argument order. The rest of the application only ever imports and depends on your interface, never the legacy SDK directly — which is what makes a future vendor swap a one-file change instead of a codebase-wide search-and-replace.

**When to use it:** Integrating vendor SDKs, migrating from one storage backend to another without touching call sites.

**Trade-offs:** The adapter absorbs 100% of the vendor-specific naming and argument quirks in one place, so nothing else in the codebase needs to know the legacy SDK exists — the cost is a thin translation layer that adds no new behavior of its own, just indirection, so it's only worth it when you actually expect to swap or mock the underlying implementation.

**Learn more:** [Refactoring.Guru: Design Patterns](https://refactoring.guru/design-patterns)


In [ ]:
# --- Legacy SDK (imagine this is a third-party library you can't change) ---
class LegacyStorageSDK:
    def uploadBlob(self, containerName, blobName, data: bytes): # camelCase API
        print(f"[Legacy] Uploading {blobName} to {containerName}")

    def deleteBlob(self, containerName, blobName):
        print(f"[Legacy] Deleting {blobName} from {containerName}")

    def fetchBlob(self, containerName, blobName) -> bytes:
        print(f"[Legacy] Fetching {blobName} from {containerName}")
        return b"<data>"

# --- Our application's expected interface ---
class ObjectStore:
    def put(self, bucket: str, key: str, data: bytes): ...
    def delete(self, bucket: str, key: str): ...
    def get(self, bucket: str, key: str) -> bytes: ...

# --- Adapter ---
class S3Adapter(ObjectStore):
    def __init__(self, sdk: LegacyStorageSDK):
        self._sdk = sdk

    def put(self, bucket, key, data):
        self._sdk.uploadBlob(bucket, key, data)   # translate snake_case → camelCase

    def delete(self, bucket, key):
        self._sdk.deleteBlob(bucket, key)

    def get(self, bucket, key) -> bytes:
        return self._sdk.fetchBlob(bucket, key)

# Time: O(1) wrapper overhead
# Space: O(1) — no data duplication

store: ObjectStore = S3Adapter(LegacyStorageSDK())
store.put("my-bucket", "report.pdf", b"%PDF...")
data = store.get("my-bucket", "report.pdf")
store.delete("my-bucket", "report.pdf")

#### Practice — Design Patterns

Choose **one** of the following, implement it cold, then compare with the reference above:

1. **Factory** — extend `NotifierFactory` to support a `WebhookNotifier` that POSTs to a URL. Register it without changing existing code.
2. **Observer** — add a `once(event, callback)` method to `EventBus` that auto-unsubscribes after the first emit.
3. **Strategy** — write a `Paginator` that accepts a `fetch_page(page_num) → List[dict]` strategy and iterates all pages lazily.
4. **Decorator** — write a `retry(max_attempts, exceptions)` decorator that re-calls the function on specified exceptions with exponential back-off.

Annotate all implementations with `# Time:` and `# Space:`.

In [ ]:
# Your practice implementation here

---
# 3. System Design Basics

### Study checklist
- [ ] Stateless services and load balancing
- [ ] Caching — LRU and TTL patterns
- [ ] Rate limiting — token bucket
- [ ] CAP theorem — consistency vs availability trade-off
- [ ] Consistent hashing — minimise reshuffling on scale-out

### Notes
Write design decisions, gotchas, and edge cases here.

### Beginner-friendly intro
System design is about choosing the right trade-offs under real-world constraints: load, latency, cost, and failure. At principal level you are expected to decompose ambiguous requirements, name and justify trade-offs, and know when a simple solution beats a clever one.

#### 3.1 Stateless services and load balancing

**Approach:** Workers hold no request-local state; a load balancer distributes across them. Round-robin is the simplest strategy. Least-connections or consistent hashing are better for uneven workloads.

**When to use it:** Any web tier, RPC service, or batch worker fleet.

**Trade-offs:** Easy horizontal scale; sessions must live in shared storage (Redis, DB), not in-process.

**Learn more:** [System Design Primer](https://github.com/donnemartin/system-design-primer)


In [ ]:
import itertools
from typing import Callable

class Worker:
    def __init__(self, wid: str):
        self.id = wid
        self.requests_handled = 0

    def handle(self, request: str) -> str:
        self.requests_handled += 1
        return f"[{self.id}] handled: {request}"

class RoundRobinBalancer:
    def __init__(self, workers: list[Worker]):
        self._pool = itertools.cycle(workers)
        self._workers = workers

    def dispatch(self, request: str) -> str:
        worker = next(self._pool)
        return worker.handle(request)

    def stats(self) -> dict:
        return {w.id: w.requests_handled for w in self._workers}

# Time: O(1) per dispatch
# Space: O(n) for the worker pool

workers = [Worker(f"W{i}") for i in range(3)]
lb = RoundRobinBalancer(workers)
for i in range(9):
    lb.dispatch(f"req-{i}")
print(lb.stats())   # evenly distributed: {W0:3, W1:3, W2:3}

#### 3.2 LRU Cache

**Approach:** `OrderedDict` preserves insertion order and supports O(1) move-to-end. On get: move to end (most-recently-used). On set: evict the first item (LRU) when over capacity.

**When to use it:** DB query results, computed thumbnails, session tokens — any hot read with bounded memory.

**Trade-offs:** O(1) get/set; needs a separate TTL layer for expiry; not thread-safe without a lock.

**Learn more:** [System Design Primer](https://github.com/donnemartin/system-design-primer)


In [ ]:
from collections import OrderedDict
from typing import Optional, Any
import threading

class LRUCache:
    def __init__(self, capacity: int):
        if capacity < 1:
            raise ValueError("capacity must be >= 1")
        self._cap   = capacity
        self._cache: OrderedDict[Any, Any] = OrderedDict()
        self._lock  = threading.Lock()

    def get(self, key) -> Optional[Any]:
        with self._lock:
            if key not in self._cache:
                return None
            self._cache.move_to_end(key)   # mark as recently used
            return self._cache[key]

    def put(self, key, value) -> None:
        with self._lock:
            if key in self._cache:
                self._cache.move_to_end(key)
            self._cache[key] = value
            if len(self._cache) > self._cap:
                self._cache.popitem(last=False)   # evict LRU (front)

    def __repr__(self):
        return f"LRUCache({dict(self._cache)})"

# Time: O(1) get and put
# Space: O(capacity)

cache = LRUCache(3)
for k, v in [("a",1),("b",2),("c",3)]:
    cache.put(k, v)
cache.get("a")        # 'a' becomes most-recent
cache.put("d", 4)     # evicts 'b' (LRU)
print(cache)          # LRUCache({'c':3, 'a':1, 'd':4})

#### 3.3 Token Bucket Rate Limiter

**Approach:** Each client has a bucket of `capacity` tokens that refills at `rate` tokens/second. Each request consumes one token. Requests that find an empty bucket are rejected.

**When to use it:** API gateways, webhook senders, outbound email/SMS flows.

**Trade-offs:** Allows short bursts up to `capacity`; sliding-window log is more accurate but O(n) memory.

**Learn more:** [System Design Primer](https://github.com/donnemartin/system-design-primer)


In [ ]:
import time

class TokenBucket:
    def __init__(self, capacity: int, rate: float):
        self._capacity  = capacity   # max tokens (burst ceiling)
        self._rate      = rate       # tokens added per second
        self._tokens    = float(capacity)
        self._last_refill = time.monotonic()

    def _refill(self) -> None:
        now     = time.monotonic()
        elapsed = now - self._last_refill
        self._tokens = min(
            self._capacity,
            self._tokens + elapsed * self._rate
        )
        self._last_refill = now

    def allow(self, cost: int = 1) -> bool:
        self._refill()
        if self._tokens >= cost:
            self._tokens -= cost
            return True
        return False

    def __repr__(self):
        return f"TokenBucket(tokens={self._tokens:.2f}/{self._capacity}, rate={self._rate}/s)"

# Time: O(1) per allow() call
# Space: O(1) per client bucket

bucket = TokenBucket(capacity=5, rate=2)   # 5-token burst, 2/s refill
results = [bucket.allow() for _ in range(7)]
print(results)     # [T,T,T,T,T,F,F] — first 5 pass, then empty
time.sleep(1)
print(bucket.allow())   # True — ~2 tokens refilled

#### 3.4 CAP Theorem

**Approach:** In a partitioned network, a distributed system can be Consistent (every read sees the latest write) or Available (every request gets a response) — not both simultaneously. CP systems wait for consensus; AP systems return stale data rather than error.

**When to use it:** Choose CP for financial ledgers, inventory counts; AP for shopping carts, DNS, analytics.

**Trade-offs:** No escape from the theorem — the choice is which guarantee to drop under partition.

**Learn more:** [System Design Primer](https://github.com/donnemartin/system-design-primer)


In [ ]:
import time

class CPStore:
    # Consistent + Partition-tolerant: refuses reads during partition
    def __init__(self):
        self._data = {}
        self._partitioned = False

    def write(self, key, value):
        self._data[key] = value

    def read(self, key):
        if self._partitioned:
            raise ConnectionError("Partition active — refusing stale read (CP)")
        return self._data.get(key)

    def simulate_partition(self, on: bool):
        self._partitioned = on

class APStore:
    # Available + Partition-tolerant: returns possibly stale data
    def __init__(self):
        self._data = {}
        self._partitioned = False
        self._stale_data = {}

    def write(self, key, value):
        self._data[key] = value
        if not self._partitioned:
            self._stale_data[key] = value   # sync replica when healthy

    def read(self, key):
        if self._partitioned:
            print(f"  [AP] Partition active — returning possibly stale value")
            return self._stale_data.get(key)   # serve old replica
        return self._data.get(key)

    def simulate_partition(self, on: bool):
        self._partitioned = on

# Time/Space: O(k) for k keys -- illustrative only; real stores are far more complex

for StoreClass in (CPStore, APStore):
    s = StoreClass()
    s.write("balance", 1000)
    s.simulate_partition(True)
    try:
        print(f"{StoreClass.__name__}: {s.read('balance')}")
    except ConnectionError as e:
        print(f"{StoreClass.__name__}: {e}")

#### 3.5 Consistent Hashing

**Approach:** Map both nodes and keys onto a ring (hash mod 2^32). Each key is owned by the first node clockwise from its hash. Virtual nodes smooth out uneven distribution. On node add/remove, only O(k/n) keys remapped.

**When to use it:** Distributed caches (Memcached, Redis Cluster), sharded databases, CDN routing.

**Trade-offs:** Far fewer key migrations than modular hashing; slightly more complex implementation.

**Learn more:** [System Design Primer](https://github.com/donnemartin/system-design-primer)


In [ ]:
import hashlib, bisect

class ConsistentHashRing:
    def __init__(self, vnodes: int = 150):
        self._vnodes   = vnodes
        self._ring:    list[int] = []
        self._node_map: dict[int, str] = {}

    def _hash(self, key: str) -> int:
        return int(hashlib.sha1(key.encode()).hexdigest(), 16)

    def add_node(self, node: str) -> None:
        for i in range(self._vnodes):
            h = self._hash(f"{node}#{i}")
            bisect.insort(self._ring, h)
            self._node_map[h] = node

    def remove_node(self, node: str) -> None:
        for i in range(self._vnodes):
            h = self._hash(f"{node}#{i}")
            self._ring.remove(h)
            del self._node_map[h]

    def get_node(self, key: str) -> str:
        if not self._ring:
            raise RuntimeError("Ring is empty")
        h = self._hash(key)
        idx = bisect.bisect_right(self._ring, h) % len(self._ring)
        return self._node_map[self._ring[idx]]

# Time: O(v log(n*v)) add/remove, O(log(n*v)) lookup
#       where n = nodes, v = vnodes
# Space: O(n * v) for ring + map

ring = ConsistentHashRing(vnodes=100)
for srv in ("cache-1", "cache-2", "cache-3"):
    ring.add_node(srv)

keys = [f"user:{i}" for i in range(12)]
from collections import Counter
before = {k: ring.get_node(k) for k in keys}
dist = Counter(before.values())
print("Before removal:", dict(dist))

ring.remove_node("cache-2")
after = {k: ring.get_node(k) for k in keys}
dist2 = Counter(after.values())
print("After  removal:", dict(dist2))
print("Keys moved:", sum(1 for k in keys if before[k] != after[k]))

#### Practice — System Design

Pick one problem and implement a clean solution:

1. **Write-through cache** — extend `LRUCache` with a `write_through(key, value, store)` method that updates both the cache and a backing store atomically.
2. **Sliding window rate limiter** — implement a rate limiter using a `collections.deque` of timestamps instead of token bucket. Compare memory trade-offs.
3. **Weighted round-robin** — modify `RoundRobinBalancer` so each worker has a weight; high-weight workers receive proportionally more requests.
4. **Ring rebalance report** — given a `ConsistentHashRing`, add a method that returns which keys would migrate if a new node were added, *without actually adding it*.

Annotate all implementations with `# Time:` and `# Space:`.

In [ ]:
# Your practice implementation here

---
# 4. OS Basics

### Study checklist
- [ ] Processes vs threads — GIL impact in Python
- [ ] Stack vs heap memory
- [ ] Deadlock — conditions and prevention
- [ ] File descriptors and I/O
- [ ] CPU scheduling — round-robin simulation

### Notes
Write design decisions, gotchas, and edge cases here.

### Beginner-friendly intro
OS concepts underpin every system you build: why threads don't speed up CPU-bound Python, why a long-running service leaks file handles, how a simple lock ordering rule prevents deadlocks. Interviewers use OS questions to test whether you reason from first principles.

#### 4.1 Processes vs Threads — GIL impact

**Approach:** Python's GIL lets only one thread execute Python bytecode at a time. For I/O-bound work (network, disk), threads still help because the GIL is released during I/O. For CPU-bound work, use `multiprocessing` — each process has its own GIL.

**When to use it:** Threads for concurrent I/O (web scraping, DB queries). Processes for parallel computation (ML inference, image processing).

**Trade-offs:** Threads share memory (cheap comms, race risks). Processes isolate memory (safe, slower IPC).

**Learn more:** [GeeksforGeeks: Operating Systems Tutorial](https://www.geeksforgeeks.org/operating-systems/operating-systems/)


In [ ]:
import pickle
import threading, multiprocessing, time

def cpu_task(n: int) -> int:
    # Pure computation -- GIL-bound regardless of how many threads run it
    return sum(i * i for i in range(n))

def io_task(label: str) -> None:
    # Simulated I/O -- GIL released during sleep, so threads genuinely help here
    time.sleep(0.05)

def main() -> None:
    N = 10

    # --- Threaded I/O (should be fast) ---
    t0 = time.perf_counter()
    threads = [threading.Thread(target=io_task, args=(f"t{i}",)) for i in range(N)]
    for t in threads: t.start()
    for t in threads: t.join()
    threaded_io_ms = (time.perf_counter() - t0) * 1000
    print(f"Threaded I/O:        {threaded_io_ms:.0f} ms")

    # --- Sequential I/O (baseline) ---
    t0 = time.perf_counter()
    for i in range(N): io_task(f"s{i}")
    sequential_io_ms = (time.perf_counter() - t0) * 1000
    print(f"Sequential I/O:      {sequential_io_ms:.0f} ms  (threads ~{sequential_io_ms/threaded_io_ms:.0f}x faster)")

    # --- Threaded CPU-bound work (GIL held throughout: no speedup expected) ---
    t0 = time.perf_counter()
    cpu_threads = [threading.Thread(target=cpu_task, args=(500_000,)) for _ in range(N)]
    for t in cpu_threads: t.start()
    for t in cpu_threads: t.join()
    threaded_cpu_ms = (time.perf_counter() - t0) * 1000
    print(f"Threaded CPU-bound:  {threaded_cpu_ms:.0f} ms  (no speedup -- only one thread runs Python bytecode at a time)")

    # --- Multiprocessing CPU (bypasses the GIL: each process has its own interpreter) ---
    # Must run inside `if __name__ == "__main__":` -- on Windows (and most
    # Jupyter/interactive sessions) the spawn start method re-imports this
    # module in every worker process; without the guard, that reimport would
    # re-run this whole block again in each child, recursively spawning more
    # pools. It can still fail when run interactively (e.g. directly in a
    # Jupyter kernel on Windows), since spawn needs to pickle cpu_task from a
    # real, importable module rather than the kernel's live __main__ namespace
    # -- that's a genuine platform limitation of multiprocessing, not a bug here.
    t0 = time.perf_counter()
    try:
        with multiprocessing.Pool(processes=4) as pool:
            pool.map(cpu_task, [500_000] * N)
        mp_cpu_ms = (time.perf_counter() - t0) * 1000
        print(f"MP CPU (4 procs):    {mp_cpu_ms:.0f} ms  (bypasses the GIL -- true parallelism)")
    except (pickle.PicklingError, AttributeError) as e:
        print(f"MP CPU (4 procs):    skipped -- {type(e).__name__}: multiprocessing.Pool needs to "
              f"pickle worker functions from a real, importable module; this fails when run "
              f"interactively (e.g. directly in a Jupyter kernel on Windows). Save this code to a "
              f".py file and import cpu_task from it to see the real speedup.")

if __name__ == "__main__":
    main()

# Time: I/O tasks O(tasks/threads) wall-clock; threaded CPU tasks O(tasks) wall-clock (GIL-serialized);
#       multiprocessing CPU tasks O(tasks/processes) wall-clock
# Space: O(n) threads or processes in memory


#### 4.2 Stack vs Heap Memory

**Approach:** The stack holds call frames and local variables (fast, fixed-size, auto-freed on return). The heap holds dynamically allocated objects (flexible size, garbage collected). In Python everything is heap-allocated; `tracemalloc` measures it.

**When to use it:** Stack overflow diagnosis (deep recursion), memory leak hunting, GC tuning.

**Trade-offs:** Stack allocation is O(1) and implicit; heap requires GC, fragmentation is possible.

**Learn more:** [GeeksforGeeks: Operating Systems Tutorial](https://www.geeksforgeeks.org/operating-systems/operating-systems/)


In [ ]:
import tracemalloc

tracemalloc.start()

def stack_heavy(n: int) -> int:
    # Each recursive call adds a frame to the call stack
    if n == 0: return 0
    return n + stack_heavy(n - 1)

def heap_heavy(n: int) -> list:
    # Allocates a large list on the heap
    return list(range(n))

snap1 = tracemalloc.take_snapshot()
result_heap = heap_heavy(100_000)
snap2 = tracemalloc.take_snapshot()

top_stats = snap2.compare_to(snap1, 'lineno')
for stat in top_stats[:3]:
    print(stat)

import sys
result_stack = stack_heavy(50)   # small n to avoid RecursionError
print(f"stack_heavy result: {result_stack}")
print(f"heap list size in memory: {sys.getsizeof(result_heap):,} bytes")

tracemalloc.stop()
# Time: O(n) for both functions
# Space: stack_heavy O(n) call frames; heap_heavy O(n) list

#### 4.3 Deadlock Prevention — Lock Ordering

**Approach:** Deadlock requires four conditions (mutual exclusion, hold-and-wait, no preemption, circular wait). Breaking circular wait by always acquiring locks in a consistent global order (e.g. sorted by `id()`) prevents deadlock.

**When to use it:** Any code that acquires multiple locks in the same scope.

**Trade-offs:** Simple and provably safe; requires discipline across the entire codebase; lock IDs must be stable.

**Learn more:** [GeeksforGeeks: Operating Systems Tutorial](https://www.geeksforgeeks.org/operating-systems/operating-systems/)


In [ ]:
import threading, time

def transfer_unsafe(src, dst, amount):
    # UNSAFE: no lock order -> circular wait possible
    with src['lock']:
        time.sleep(0.001)          # simulate work -- raises deadlock probability
        with dst['lock']:
            src['balance'] -= amount
            dst['balance'] += amount

def transfer_safe(src, dst, amount):
    # SAFE: always acquire the lower-id lock first
    first, second = (src, dst) if id(src) < id(dst) else (dst, src)
    with first['lock']:
        with second['lock']:
            src['balance'] -= amount
            dst['balance'] += amount

def make_account(balance):
    return {'balance': balance, 'lock': threading.Lock()}

# Demo safe transfers -- run many concurrent transfers, check no deadlock
alice = make_account(1000)
bob   = make_account(1000)

threads = []
for _ in range(50):
    threads.append(threading.Thread(target=transfer_safe, args=(alice, bob, 10)))
    threads.append(threading.Thread(target=transfer_safe, args=(bob, alice, 10)))

for t in threads: t.start()
for t in threads: t.join()

print(f"Alice: {alice['balance']}, Bob: {bob['balance']}")  # sum always 2000

# Time: O(1) per transfer (lock acquisition is O(1) amortised)
# Space: O(1) extra per transfer

#### 4.4 File Descriptors and Safe I/O

**Approach:** Every `open()` consumes an OS file descriptor (FD). Unclosed FDs leak until the process hits its FD limit. Use context managers (`with open(...)`) or a custom `SafeFile` wrapper to guarantee `.close()` even on exceptions.

**When to use it:** Any file, socket, or pipe operation in a long-running service.

**Trade-offs:** Context managers add one level of indentation; far better than resource leaks in production.

**Learn more:** [GeeksforGeeks: Operating Systems Tutorial](https://www.geeksforgeeks.org/operating-systems/operating-systems/)


In [ ]:
import os, tempfile

class SafeFile:
    # Explicit context manager -- guarantees close() on exit or exception
    def __init__(self, path: str, mode: str = 'r'):
        self._path = path
        self._mode = mode
        self._fh   = None

    def __enter__(self):
        self._fh = open(self._path, self._mode)
        return self._fh

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self._fh:
            self._fh.close()
        return False   # do not suppress exceptions

# Time: O(1) open/close; O(n) read/write for n bytes
# Space: O(n) for read buffer; O(1) for streaming writes

with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as tmp:
    path = tmp.name
    tmp.write("hello from safe file\n")

with SafeFile(path, 'r') as f:
    content = f.read()
    print(f"Read {len(content)} chars: {content.strip()!r}")

# Verify closed
try:
    f.read()   # should raise ValueError: I/O operation on closed file
except ValueError as e:
    print(f"Correctly closed: {e}")

os.unlink(path)

#### 4.5 CPU Scheduling — Round Robin Simulation

**Approach:** Processes share the CPU in fixed time quanta. A ready queue cycles through them; a process that doesn't finish in its quantum goes to the back.

**When to use it:** Understanding preemptive multitasking, async event loops, thread scheduling fairness.

**Trade-offs:** Fair, prevents starvation; context-switch overhead; short-burst processes penalised if quantum is large.

**Learn more:** [GeeksforGeeks: Operating Systems Tutorial](https://www.geeksforgeeks.org/operating-systems/operating-systems/)


In [ ]:
from collections import deque
from dataclasses import dataclass, field
from typing import List

@dataclass
class Process:
    pid:        str
    burst:      int          # total CPU time needed (ms)
    remaining:  int = field(init=False)
    start_time: int = field(init=False, default=-1)
    end_time:   int = field(init=False, default=-1)

    def __post_init__(self):
        self.remaining = self.burst

def round_robin(processes: List[Process], quantum: int) -> None:
    queue  = deque(processes)
    clock  = 0
    log    = []
    while queue:
        p = queue.popleft()
        if p.start_time == -1:
            p.start_time = clock
        run_for  = min(quantum, p.remaining)
        p.remaining -= run_for
        clock      += run_for
        log.append(f"t={clock:>4}: {p.pid} ran {run_for}ms (rem={p.remaining})")
        if p.remaining > 0:
            queue.append(p)   # not done — back of queue
        else:
            p.end_time = clock

    print("\n".join(log))
    print("\nSummary:")
    for p in processes:
        turnaround = p.end_time - p.start_time
        waiting    = turnaround - p.burst
        print(f"  {p.pid}: turnaround={turnaround}ms  waiting={waiting}ms")

# Time: O(sum(burst) / quantum) iterations
# Space: O(n) ready queue

procs = [Process("P1", 24), Process("P2", 3), Process("P3", 3)]
round_robin(procs, quantum=4)

#### Practice — OS Basics

Pick one:

1. **Thread pool** — implement a minimal `ThreadPool(n_workers)` with a `submit(fn, *args)` method and `shutdown()`. Use `queue.Queue` as the task buffer.
2. **Recursive FD checker** — write a context manager that asserts no new file descriptors are leaked after a `with` block (use `os.getpid()` + `/proc/self/fd` on Linux or `resource.getrlimit`).
3. **Priority scheduling** — extend the round-robin scheduler to `SRTF` (shortest remaining time first): always run the process with least remaining time.
4. **Semaphore from scratch** — implement a `Semaphore(n)` using only `threading.Lock` and `threading.Condition`. Verify with a bounded-buffer producer-consumer test.

Annotate all implementations with `# Time:` and `# Space:`.

In [ ]:
# Your practice implementation here

---
# 5. DBMS

### Study checklist
- [ ] ACID properties — atomicity, consistency, isolation, durability
- [ ] B-tree index — how range scans work
- [ ] JOIN types — inner, left, self
- [ ] Window functions — RANK and running totals
- [ ] Normalisation — 1NF → 2NF → 3NF

### Notes
Write design decisions, gotchas, and edge cases here.

### Beginner-friendly intro
Databases are the memory of your system. ACID gives you correctness guarantees; indexes give you speed; normalisation gives you consistency. At principal level, know the cost of each abstraction: a JOIN on un-indexed columns, an un-batched INSERT loop, a SELECT * in a hot path.

#### 5.1 ACID — Write-Ahead Log and Rollback

**Approach:** Atomicity: all-or-nothing. Consistency: schema rules hold. Isolation: concurrent transactions don't corrupt each other. Durability: committed data survives crashes. A write-ahead log (WAL) records the intent before the mutation; on crash, replay or roll back from the log.

**When to use it:** Financial transfers, order placement, any multi-step mutation that must succeed or fail as a unit.

**Trade-offs:** Durability requires fsync (expensive I/O); weaker isolation levels trade correctness for throughput.

**Learn more:** [GeeksforGeeks: DBMS Tutorial](https://www.geeksforgeeks.org/dbms/dbms/)


In [ ]:
class SimpleDB:
    # Minimal in-memory DB illustrating WAL + rollback
    def __init__(self):
        self._store: dict = {}
        self._wal:   list = []         # write-ahead log: list of (op, key, old, new)
        self._in_tx: bool = False

    def begin(self):
        if self._in_tx: raise RuntimeError("Already in transaction")
        self._wal.clear()
        self._in_tx = True

    def set(self, key, value):
        if not self._in_tx: raise RuntimeError("No active transaction")
        old = self._store.get(key)
        self._wal.append(('set', key, old, value))  # log before mutation
        self._store[key] = value                     # mutate

    def commit(self):
        if not self._in_tx: raise RuntimeError("No active transaction")
        self._wal.clear()    # log entries are no longer needed
        self._in_tx = False

    def rollback(self):
        if not self._in_tx: raise RuntimeError("No active transaction")
        for op, key, old, _ in reversed(self._wal):  # undo in reverse order
            if old is None:
                self._store.pop(key, None)
            else:
                self._store[key] = old
        self._wal.clear()
        self._in_tx = False

    def get(self, key): return self._store.get(key)
    def __repr__(self): return f"SimpleDB({self._store})"

# Time: O(n) rollback where n = WAL entries in transaction
# Space: O(n) WAL

db = SimpleDB()
db.begin()
db.set("alice", 1000)
db.set("bob", 500)
db.commit()
print("After commit:", db)

db.begin()
db.set("alice", 800)   # debit alice
db.set("bob", 700)     # credit bob
print("Mid-tx:", db)
db.rollback()          # something went wrong
print("After rollback:", db)   # back to alice=1000, bob=500

#### 5.2 B-Tree Index Analogue

**Approach:** A B-tree keeps keys sorted. Binary search gives O(log n) point lookups; range scans are O(log n + k). We simulate the sorted-key property with Python's `bisect` module.

**When to use it:** Any column appearing in WHERE, ORDER BY, or JOIN ON clauses.

**Trade-offs:** Indexes speed reads but slow writes (every INSERT updates the index). Full-table scans beat indexed reads when selectivity is low.

**Learn more:** [GeeksforGeeks: DBMS Tutorial](https://www.geeksforgeeks.org/dbms/dbms/)


In [ ]:
import bisect
from typing import Any, Optional

class SortedIndex:
    # Simulates a B-tree leaf-level sorted index
    def __init__(self):
        self._keys:   list = []
        self._values: list = []

    def insert(self, key, value) -> None:
        pos = bisect.bisect_left(self._keys, key)
        self._keys.insert(pos, key)
        self._values.insert(pos, value)

    def lookup(self, key) -> Optional[Any]:
        # Point lookup O(log n)
        pos = bisect.bisect_left(self._keys, key)
        if pos < len(self._keys) and self._keys[pos] == key:
            return self._values[pos]
        return None

    def range_scan(self, lo, hi) -> list:
        # Range scan O(log n + k) where k = matching rows
        left  = bisect.bisect_left(self._keys, lo)
        right = bisect.bisect_right(self._keys, hi)
        return list(zip(self._keys[left:right], self._values[left:right]))

# Time: O(n) insert (list shift); O(log n) lookup; O(log n + k) range
# Space: O(n) for n indexed rows

idx = SortedIndex()
for salary, name in [(80000,"Alice"),(95000,"Bob"),(70000,"Carol"),
                     (110000,"Dave"),(85000,"Eve")]:
    idx.insert(salary, name)

print("Lookup 95000:", idx.lookup(95000))
print("Range 80000-100000:", idx.range_scan(80000, 100000))

#### 5.3 JOIN Types — Inner, Left, Self

**Approach:** INNER JOIN — rows with matching keys in both tables. LEFT JOIN — all rows from left, NULL-filled if no match on right. SELF JOIN — a table joined with itself (e.g. employee/manager).

**When to use it:** Reporting, enriching events with reference data, hierarchical queries.

**Trade-offs:** Hash join is O(n+m) with O(n) space; nested-loop is O(n*m) but O(1) space — the DB optimizer chooses.

**Learn more:** [GeeksforGeeks: DBMS Tutorial](https://www.geeksforgeeks.org/dbms/dbms/)


In [ ]:
from typing import List, Dict, Any

Row = Dict[str, Any]

def inner_join(left: List[Row], right: List[Row], on: str) -> List[Row]:
    # Hash join: build a hash map on right, probe with left
    idx: Dict[Any, List[Row]] = {}
    for r in right:
        idx.setdefault(r[on], []).append(r)
    result = []
    for l in left:
        for r in idx.get(l[on], []):
            result.append({**l, **r})
    return result

def left_join(left: List[Row], right: List[Row], on: str) -> List[Row]:
    idx: Dict[Any, List[Row]] = {}
    for r in right:
        idx.setdefault(r[on], []).append(r)
    result = []
    for l in left:
        matches = idx.get(l[on], [None])
        for r in matches:
            result.append({**l, **(r or {})})
    return result

def self_join(table: List[Row], key: str, ref_key: str) -> List[Row]:
    # Find the referenced row within the same table
    idx = {row[key]: row for row in table}
    result = []
    for row in table:
        ref = idx.get(row.get(ref_key))
        if ref:
            result.append({"name": row["name"], "manager": ref["name"]})
    return result

# Time: O(n+m) hash join; O(n) self join
# Space: O(m) for hash index; O(n) result

employees = [{"id":1,"name":"Alice","dept_id":10},
             {"id":2,"name":"Bob",  "dept_id":20},
             {"id":3,"name":"Carol","dept_id":10}]
departments = [{"dept_id":10,"dept":"Engineering"},
               {"dept_id":20,"dept":"Marketing"}]

print("INNER JOIN:")
for row in inner_join(employees, departments, "dept_id"):
    print(" ", row)

# Self-join: org chart
staff = [{"id":1,"name":"CEO",  "mgr_id":None},
         {"id":2,"name":"Alice","mgr_id":1},
         {"id":3,"name":"Bob",  "mgr_id":1}]
print("\nSELF JOIN (manager):")
for row in self_join(staff, "id", "mgr_id"):
    print(" ", row)

#### 5.4 Window Functions — RANK and Running Total

**Approach:** Window functions compute aggregates over a sliding 'window' of rows without collapsing them. RANK assigns a position within a partition; running SUM accumulates across the ordered partition.

**When to use it:** Leaderboards, time-series running totals, top-N per group queries.

**Trade-offs:** Computed in one SQL pass by the engine (efficient); simulating in Python requires sorting + iteration.

**Learn more:** [GeeksforGeeks: DBMS Tutorial](https://www.geeksforgeeks.org/dbms/dbms/)


In [ ]:
from typing import List, Dict, Any
import itertools

def window_rank_and_running(rows: List[Dict], partition_by: str,
                             order_by: str, value_col: str) -> List[Dict]:
    result = []
    # Group by partition
    sorted_rows = sorted(rows, key=lambda r: (r[partition_by], -r[order_by]))
    for _, group in itertools.groupby(sorted_rows, key=lambda r: r[partition_by]):
        group = list(group)
        running = 0
        for rank, row in enumerate(group, start=1):
            running += row[value_col]
            result.append({**row, "rank": rank, "running_total": running})
    return result

# Time: O(n log n) due to sort
# Space: O(n) for result

sales = [
    {"rep": "Alice", "month": 3, "revenue": 12000},
    {"rep": "Alice", "month": 2, "revenue": 9000},
    {"rep": "Bob",   "month": 3, "revenue": 15000},
    {"rep": "Bob",   "month": 2, "revenue": 11000},
    {"rep": "Alice", "month": 1, "revenue": 8000},
]
for row in window_rank_and_running(sales, "rep", "month", "revenue"):
    print(row)

#### 5.5 Normalisation — 1NF → 2NF → 3NF

**Approach:** 1NF — atomic values, no repeating groups. 2NF — no partial dependency (all non-key attrs depend on the whole PK). 3NF — no transitive dependency (non-key attrs depend only on the PK, not on each other).

**When to use it:** Relational schema design; interview schema critique questions.

**Trade-offs:** Higher normal forms reduce anomalies but add JOINs; sometimes deliberately denormalise for read performance.

**Learn more:** [GeeksforGeeks: DBMS Tutorial](https://www.geeksforgeeks.org/dbms/dbms/)


In [ ]:
# Illustrate normalisation as Python dicts/tables (no SQL engine needed)

# --- UN-NORMALISED (0NF): repeating groups ---
unnormalised = [
    {"order_id": 1, "customer": "Alice", "city": "Pune",
     "items": "Widget x2, Gadget x1", "total": 150},
    {"order_id": 2, "customer": "Bob",   "city": "Delhi",
     "items": "Widget x1",            "total": 50},
]

# --- 1NF: atomic values (one item per row) ---
nf1 = [
    {"order_id": 1, "customer": "Alice", "city": "Pune",  "item": "Widget", "qty": 2},
    {"order_id": 1, "customer": "Alice", "city": "Pune",  "item": "Gadget", "qty": 1},
    {"order_id": 2, "customer": "Bob",   "city": "Delhi", "item": "Widget", "qty": 1},
]
# PK = (order_id, item)
# Partial dependency: customer, city depend only on order_id -> violates 2NF

# --- 2NF: split out partial dependency ---
orders = [
    {"order_id": 1, "customer_id": 101},
    {"order_id": 2, "customer_id": 102},
]
customers_2nf = [
    {"customer_id": 101, "customer": "Alice", "city": "Pune"},
    {"customer_id": 102, "customer": "Bob",   "city": "Delhi"},
]
order_items = [
    {"order_id": 1, "item": "Widget", "qty": 2},
    {"order_id": 1, "item": "Gadget", "qty": 1},
    {"order_id": 2, "item": "Widget", "qty": 1},
]
# 'city' still depends on 'customer' (transitive via customer_id -> customer -> city)
# -> violates 3NF

# --- 3NF: separate customer and city ---
customers_3nf = [{"customer_id": 101, "customer": "Alice", "city_id": 1},
                 {"customer_id": 102, "customer": "Bob",   "city_id": 2}]
cities         = [{"city_id": 1, "city": "Pune"},
                  {"city_id": 2, "city": "Delhi"}]

print("3NF tables:", list(map(lambda t: len(t), [orders, order_items, customers_3nf, cities])), "rows each")
print("No partial or transitive dependencies remain.")
# Space: O(n) per table; fewer duplicated values than 0NF

#### Practice — DBMS

Pick one:

1. **Isolation levels** — extend `SimpleDB` to support READ COMMITTED: a dirty-read check that prevents reading uncommitted values from concurrent transactions.
2. **Composite index** — extend `SortedIndex` to support a 2-column composite key `(col_a, col_b)` with prefix range scans on `col_a` alone.
3. **GROUP BY + HAVING** — implement `group_by(rows, by, agg_fn, having=None)` in Python that mirrors SQL GROUP BY / HAVING semantics.
4. **BCNF decomposition** — given a schema and a set of functional dependencies, write a function that decomposes it into BCNF and outputs the resulting table schemas.

Annotate all implementations with `# Time:` and `# Space:`.

In [ ]:
# Your practice implementation here

---
# 6. Networking

### Study checklist
- [ ] TCP handshake — state machine
- [ ] HTTP/1.1 — request/response parsing and status codes
- [ ] DNS — recursive resolution
- [ ] REST — resource design and correct status codes
- [ ] TLS 1.3 — handshake overview

### Notes
Write design decisions, gotchas, and edge cases here.

### Beginner-friendly intro
Every API call, every database connection, every microservice hop is a network conversation. Knowing the layers — IP, TCP, TLS, HTTP — lets you diagnose latency, timeouts, and cert errors rather than guessing at the application layer.

#### 6.1 TCP Three-Way Handshake — State Machine

**Approach:** Client: SYN → server: SYN-ACK → client: ACK. Each step transitions state. Modelling as an explicit state machine makes invalid transitions visible immediately.

**When to use it:** Diagnosing `connection refused` vs `connection timed out`; tuning keep-alive; understanding TIME_WAIT.

**Trade-offs:** TCP guarantees ordered, reliable delivery at the cost of latency (RTT); UDP skips handshake for speed.

**Learn more:** [GeeksforGeeks: Computer Network Tutorial](https://www.geeksforgeeks.org/computer-networks/computer-network-tutorials/)


In [ ]:
from enum import Enum, auto

class TCPState(Enum):
    CLOSED      = auto()
    SYN_SENT    = auto()
    SYN_RECEIVED= auto()
    ESTABLISHED = auto()
    FIN_WAIT_1  = auto()
    FIN_WAIT_2  = auto()
    TIME_WAIT   = auto()

class TCPEndpoint:
    def __init__(self, role: str):
        self.role  = role
        self.state = TCPState.CLOSED

    def _transition(self, new_state: TCPState, event: str):
        print(f"[{self.role}] {event}: {self.state.name} -> {new_state.name}")
        self.state = new_state

    # --- client-side handshake ---
    def send_syn(self):
        assert self.state == TCPState.CLOSED
        self._transition(TCPState.SYN_SENT, "send SYN")

    def receive_syn_ack(self):
        assert self.state == TCPState.SYN_SENT
        self._transition(TCPState.ESTABLISHED, "recv SYN-ACK, send ACK")

    # --- server-side handshake ---
    def receive_syn(self):
        assert self.state == TCPState.CLOSED
        self._transition(TCPState.SYN_RECEIVED, "recv SYN, send SYN-ACK")

    def receive_ack(self):
        assert self.state == TCPState.SYN_RECEIVED
        self._transition(TCPState.ESTABLISHED, "recv ACK")

    # --- teardown (client side) ---
    def send_fin(self):
        assert self.state == TCPState.ESTABLISHED
        self._transition(TCPState.FIN_WAIT_1, "send FIN")

# Time: O(1) per state transition
# Space: O(1)

client = TCPEndpoint("CLIENT")
server = TCPEndpoint("SERVER")

client.send_syn()
server.receive_syn()
client.receive_syn_ack()
server.receive_ack()
print(f"Connection up: client={client.state.name}, server={server.state.name}")
client.send_fin()

#### 6.2 HTTP/1.1 Parsing and Status Codes

**Approach:** An HTTP/1.1 request is a start-line, headers, blank line, optional body. Parse with `splitlines()` and header splitting — no regex required for the basic case.

**When to use it:** Writing test harnesses, debugging raw HTTP traffic, implementing lightweight HTTP utilities.

**Trade-offs:** HTTP/1.1 is human-readable and cacheable; HTTP/2 multiplexes frames but is binary.

**Learn more:** [MDN: HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP)


In [ ]:
from dataclasses import dataclass
from typing import Dict, Optional

@dataclass
class HttpRequest:
    method: str
    path: str
    version: str
    headers: Dict[str, str]
    body: Optional[str]

STATUS_CODES = {
    200: "OK",              201: "Created",
    204: "No Content",     301: "Moved Permanently",
    400: "Bad Request",    401: "Unauthorized",
    403: "Forbidden",      404: "Not Found",
    409: "Conflict",       422: "Unprocessable Entity",
    429: "Too Many Requests",
    500: "Internal Server Error",
    503: "Service Unavailable",
}

def parse_http_request(raw: str) -> HttpRequest:
    lines = raw.strip().splitlines()
    method, path, version = lines[0].split(" ", 2)
    headers = {}
    i = 1

    while i < len(lines) and lines[i]:
        key, _, value = lines[i].partition(": ")
        headers[key.lower()] = value.strip()
        i += 1

    body = "\n".join(lines[i + 1:]) if i + 1 < len(lines) else None
    return HttpRequest(method, path, version, headers, body)

def format_response(
    status: int,
    body: str = "",
    content_type: str = "application/json"
) -> str:
    reason = STATUS_CODES.get(status, "Unknown")
    headers = (
        f"Content-Type: {content_type}\n"
        f"Content-Length: {len(body.encode())}\n"
    )
    return f"HTTP/1.1 {status} {reason}\n{headers}\n{body}"

# Time: O(n) for n bytes in raw request
# Space: O(h + b) where h = header count, b = body size

raw = (
    "POST /orders HTTP/1.1\n"
    "Host: api.example.com\n"
    "Content-Type: application/json\n"
    "\n"
    '{"item":"widget","qty":2}'
)

req = parse_http_request(raw)
print(req)
print(format_response(201, '{"order_id": "ORD-99"}'))


#### 6.3 DNS Recursive Resolution

**Approach:** DNS resolves names bottom-up: root → TLD → authoritative. Each level caches the answer for the TTL. `functools.lru_cache` simulates the resolver cache.

**When to use it:** Diagnosing DNS propagation delays, TTL-induced stale records, split-horizon DNS.

**Trade-offs:** Caching speeds resolution dramatically; stale cache causes hard-to-debug 'it works on my machine' issues.

**Learn more:** [GeeksforGeeks: Computer Network Tutorial](https://www.geeksforgeeks.org/computer-networks/computer-network-tutorials/)


In [ ]:
import functools, time

# Simulated authoritative data
DNS_RECORDS = {
    ".":                  {"ns": "root-ns."},
    "com.":               {"ns": "gtld-ns."},
    "example.com.":       {"ns": "ns1.example.com."},
    "api.example.com.":   {"A": "93.184.216.34", "ttl": 300},
    "www.example.com.":   {"A": "93.184.216.35", "ttl": 3600},
}

query_count = 0   # counts actual upstream queries (cache misses)

@functools.lru_cache(maxsize=256)
def resolve(fqdn: str) -> str:
    global query_count
    query_count += 1
    record = DNS_RECORDS.get(fqdn.rstrip(".") + ".")
    if record and "A" in record:
        return record["A"]
    # Recurse: try stripping leftmost label
    parts = fqdn.split(".", 1)
    if len(parts) == 1:
        raise ValueError(f"Could not resolve {fqdn}")
    return resolve(parts[1])

# Time: O(d) per resolution where d = domain depth; O(1) on cache hit
# Space: O(c) cache entries

for host in ["api.example.com", "www.example.com", "api.example.com"]:
    print(f"{host:30s} -> {resolve(host)}")

print(f"\nTotal upstream queries (cache misses): {query_count}")  # should be 2, not 3

#### 6.4 REST Resource Design

**Approach:** Resources are nouns, HTTP methods are verbs. GET retrieves (idempotent, cacheable). POST creates (returns 201 + Location). PUT replaces (idempotent). PATCH partially updates. DELETE removes (idempotent, returns 204).

**When to use it:** Any public or internal HTTP API. Correct status codes matter for client retry logic and CDN caching.

**Trade-offs:** REST is simple and human-debuggable; GraphQL is better for heterogeneous clients with bandwidth constraints.

**Learn more:** [MDN: HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP)


In [ ]:
from typing import Dict, Optional, Any
import uuid

class OrderStore:
    def __init__(self):
        self._orders: Dict[str, dict] = {}

    def create(self, payload: dict) -> tuple[int, dict]:
        # POST /orders -> 201 Created
        oid = str(uuid.uuid4())[:8]
        order = {"id": oid, "status": "pending", **payload}
        self._orders[oid] = order
        return 201, {"location": f"/orders/{oid}", "order": order}

    def get(self, oid: str) -> tuple[int, dict]:
        # GET /orders/{id} -> 200 OK or 404
        if oid not in self._orders:
            return 404, {"error": f"Order {oid} not found"}
        return 200, self._orders[oid]

    def update(self, oid: str, patch: dict) -> tuple[int, dict]:
        # PATCH /orders/{id} -> 200 OK or 404 or 409
        if oid not in self._orders:
            return 404, {"error": "Not found"}
        order = self._orders[oid]
        if order["status"] == "shipped" and "status" in patch:
            return 409, {"error": "Cannot change status of shipped order"}
        order.update(patch)
        return 200, order

    def delete(self, oid: str) -> tuple[int, Optional[dict]]:
        # DELETE /orders/{id} -> 204 No Content or 404
        if oid not in self._orders:
            return 404, {"error": "Not found"}
        del self._orders[oid]
        return 204, None

# Time: O(1) per operation (dict)
# Space: O(n) for n orders

store = OrderStore()
status, resp = store.create({"item": "widget", "qty": 2})
print(f"POST  -> {status}", resp)
oid = resp["order"]["id"]

status, resp = store.get(oid)
print(f"GET   -> {status}", resp)

status, resp = store.update(oid, {"status": "shipped"})
print(f"PATCH -> {status}", resp)

status, resp = store.delete(oid)
print(f"DELETE-> {status}")   # 204, no body

status, resp = store.get(oid)
print(f"GET   -> {status}", resp)   # 404

#### 6.5 TLS 1.3 Handshake Overview

**Approach:** TLS 1.3 cuts to one round-trip: ClientHello (key share + cipher list) → ServerHello (key share + cert) → client verifies cert + sends Finished → connection open. The key exchange uses ECDHE; the symmetric cipher is AES-GCM or ChaCha20-Poly1305.

**When to use it:** Every HTTPS connection. Key for diagnosing cert errors, HSTS, mixed content.

**Trade-offs:** TLS 1.3 removed weak ciphers (RC4, 3DES, RSA key exchange); 0-RTT resumption is fast but replay-vulnerable.

**Learn more:** [GeeksforGeeks: Computer Network Tutorial](https://www.geeksforgeeks.org/computer-networks/computer-network-tutorials/)


In [ ]:
# Conceptual TLS 1.3 handshake simulation (no real crypto)
import hashlib, os

def ecdhe_keygen():
    # Simulate: in reality uses elliptic-curve Diffie-Hellman
    private = os.urandom(32)
    public  = hashlib.sha256(private).digest()   # stand-in for EC point
    return private, public

def derive_shared_secret(my_private, their_public):
    # In reality: EC scalar multiplication
    return hashlib.sha256(my_private + their_public).digest()

# --- Handshake ---
client_priv, client_pub = ecdhe_keygen()
server_priv, server_pub = ecdhe_keygen()

# ClientHello: send client_pub + supported ciphers
client_hello = {
    "key_share":    client_pub.hex()[:16],
    "cipher_suites": ["TLS_AES_128_GCM_SHA256", "TLS_CHACHA20_POLY1305_SHA256"],
    "tls_versions": ["TLS 1.3"],
}

# ServerHello: send server_pub + chosen cipher + cert (omitted)
server_hello = {
    "key_share":    server_pub.hex()[:16],
    "cipher":       "TLS_AES_128_GCM_SHA256",
    "cert_subject": "CN=api.example.com",
}

# Both sides derive the same shared secret
client_secret = derive_shared_secret(client_priv, server_pub)
server_secret = derive_shared_secret(server_priv, client_pub)

print("Handshake summary:")
print("  ClientHello:", client_hello)
print("  ServerHello:", server_hello)
print("  Secrets match:", client_secret == server_secret)
print("  (Real TLS derives separate handshake + application traffic keys via HKDF)")

# Time: O(1) — real ECDHE is O(key_size^2) or O(key_size^3); negligible at 256-bit
# Space: O(1) for key material

#### Practice — Networking

Pick one:

1. **HTTP/1.1 keep-alive** — extend `HttpRequest` parsing to detect `Connection: keep-alive` and return a flag indicating whether the connection should be reused.
2. **DNS TTL expiry** — extend the `resolve` function to store `(ip, expiry)` tuples and evict entries when `time.time() > expiry`, simulating real TTL-based cache invalidation.
3. **CORS preflight** — add `options()` to `OrderStore` that returns the correct `Access-Control-Allow-*` headers for a cross-origin preflight request.
4. **TCP sliding window** — simulate a simplified TCP sliding-window flow control: a sender with `window_size` N can have N un-ACKed packets in flight; simulate ACKs arriving and the window sliding.

Annotate all implementations with `# Time:` and `# Space:`.

In [ ]:
# Your practice implementation here

---
# 7. Concurrency

### Study checklist
- [ ] Race condition and mutex (Lock)
- [ ] Semaphore — bounded resource pool
- [ ] Producer-consumer — bounded queue
- [ ] async/await — cooperative multitasking
- [ ] ThreadPoolExecutor and as_completed

### Notes
Write design decisions, gotchas, and edge cases here.

### Beginner-friendly intro
Concurrency is about managing multiple things happening at once. Threading is preemptive (OS decides when to switch); asyncio is cooperative (you choose when to yield). At principal level: choose the right model for the workload, know the failure modes, and make shared state explicit.

#### 7.1 Race Condition and Mutex

**Approach:** A race condition occurs when correctness depends on thread scheduling order. A `threading.Lock` (mutex) serialises access to the critical section. Show the unsafe version first, then the fix.

**When to use it:** Any shared mutable state: counters, caches, shared collections.

**Trade-offs:** Lock protects but reduces parallelism; prefer lock-free structures or immutability where possible.

**Learn more:** [Python docs: threading](https://docs.python.org/3/library/threading.html)


In [ ]:
import threading

class UnsafeCounter:
    def __init__(self): self.value = 0
    def increment(self):
        v = self.value       # read
        v += 1               # modify (another thread can interleave here)
        self.value = v       # write

class SafeCounter:
    def __init__(self):
        self.value = 0
        self._lock = threading.Lock()

    def increment(self):
        with self._lock:    # atomic read-modify-write
            self.value += 1

def run(counter, n_threads=100, increments=1000):
    threads = [threading.Thread(target=lambda: [counter.increment()
               for _ in range(increments)])
               for _ in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return counter.value

expected = 100 * 1000
unsafe_result = run(UnsafeCounter())
safe_result   = run(SafeCounter())

print(f"Expected:      {expected}")
print(f"Unsafe result: {unsafe_result}  (likely wrong)")
print(f"Safe result:   {safe_result}   (always correct)")

# Time: O(n*increments) operations; O(1) per increment with lock
# Space: O(n_threads) for thread objects

#### 7.2 Semaphore — Bounded Connection Pool

**Approach:** A `BoundedSemaphore(n)` allows at most n concurrent acquires. Workers that exceed the limit block until a slot is released. `BoundedSemaphore` raises if you release more times than you acquired (catches bugs).

**When to use it:** DB connection pools, HTTP connection limits, rate-limiting parallelism to protect downstream.

**Trade-offs:** Simple; threads block (waste OS resources); for async code prefer `asyncio.Semaphore`.

**Learn more:** [Python docs: threading](https://docs.python.org/3/library/threading.html)


In [ ]:
import threading, time, random

class ConnectionPool:
    def __init__(self, max_connections: int):
        self._sem = threading.BoundedSemaphore(max_connections)
        self._active = 0
        self._lock   = threading.Lock()

    def acquire(self):
        self._sem.acquire()
        with self._lock: self._active += 1

    def release(self):
        with self._lock: self._active -= 1
        self._sem.release()

    @property
    def active(self): return self._active

pool = ConnectionPool(max_connections=3)
log  = []

def worker(wid):
    pool.acquire()
    log.append(f"W{wid:02d} acquired  (active={pool.active})")
    time.sleep(random.uniform(0.01, 0.05))   # simulate query
    log.append(f"W{wid:02d} releasing (active={pool.active})")
    pool.release()

# Time: O(1) acquire/release; workers queue on the semaphore
# Space: O(max_connections) live connections

threads = [threading.Thread(target=worker, args=(i,)) for i in range(8)]
for t in threads: t.start()
for t in threads: t.join()

for line in log:
    print(line)
print("Max simultaneous connections ever exceeded 3?",
      any("active=4" in l or "active=5" in l for l in log))

#### 7.3 Producer-Consumer — Bounded Queue

**Approach:** `queue.Queue(maxsize)` is a thread-safe FIFO with blocking `put`/`get`. Producers block when the queue is full; consumers block when empty. A sentinel value (`None`) signals consumers to stop.

**When to use it:** Work queues, pipeline stages, log aggregators, event buffers.

**Trade-offs:** `queue.Queue` handles all locking internally; size cap prevents producers from overwhelming consumers.

**Learn more:** [Python docs: queue](https://docs.python.org/3/library/queue.html)


In [ ]:
import threading, queue, time, random

def producer(q: queue.Queue, n_items: int, pid: int):
    for i in range(n_items):
        item = f"P{pid}-item-{i}"
        q.put(item)   # blocks if queue full
        print(f"  produced: {item}  (qsize={q.qsize()})")
        time.sleep(random.uniform(0, 0.02))
    q.put(None)   # sentinel: this producer is done

def consumer(q: queue.Queue, cid: int):
    while True:
        item = q.get()   # blocks if queue empty
        if item is None:
            q.task_done()
            break        # graceful shutdown
        print(f"    consumed [{cid}]: {item}")
        time.sleep(random.uniform(0, 0.03))
        q.task_done()

# Time: O(n) total items processed; O(1) per put/get
# Space: O(maxsize) queue buffer

q = queue.Queue(maxsize=4)
producers = [threading.Thread(target=producer, args=(q, 3, i)) for i in range(2)]
consumers = [threading.Thread(target=consumer, args=(q, i))    for i in range(2)]

for t in consumers: t.start()
for t in producers: t.start()
for t in producers: t.join()
for t in consumers: t.join()
print("All items processed.")

#### 7.4 async/await — Cooperative Multitasking

**Approach:** `asyncio.gather` runs multiple coroutines concurrently in the same thread. Each `await` is a yield point where the event loop can run another coroutine. No threads, no locks — safe shared state; beware of CPU-bound code blocking the loop.

**When to use it:** High-concurrency I/O: web clients, async DB drivers, WebSocket servers.

**Trade-offs:** Lower overhead than threads; CPU-bound work must be offloaded with `run_in_executor`.

**Learn more:** [Python docs: asyncio](https://docs.python.org/3/library/asyncio.html)


In [ ]:
import asyncio, time

async def fetch(url: str, latency: float) -> str:
    await asyncio.sleep(latency)   # simulate network I/O
    return f"response from {url}"

async def concurrent():
    t0 = time.perf_counter()
    results = await asyncio.gather(
        fetch("api/orders",    0.1),
        fetch("api/inventory", 0.15),
        fetch("api/pricing",   0.08),
    )
    elapsed = time.perf_counter() - t0
    for r in results: print(" ", r)
    print(f"  Concurrent: {elapsed:.3f}s")  # ~0.15s (max latency, not sum)

async def sequential():
    t0 = time.perf_counter()
    for url, lat in [("api/orders",0.1),("api/inventory",0.15),("api/pricing",0.08)]:
        r = await fetch(url, lat)
        print(" ", r)
    elapsed = time.perf_counter() - t0
    print(f"  Sequential: {elapsed:.3f}s")  # ~0.33s (sum of latencies)

# Time: O(max(latencies)) concurrent vs O(sum(latencies)) sequential
# Space: O(n) for n coroutines in-flight

print("Concurrent:")
asyncio.run(concurrent())
print("Sequential:")
asyncio.run(sequential())

#### 7.5 ThreadPoolExecutor and as_completed

**Approach:** `concurrent.futures.ThreadPoolExecutor` manages a pool of threads. `submit()` returns a `Future`; `as_completed()` yields futures in completion order, not submission order. Good for heterogeneous tasks with varying durations.

**When to use it:** Parallel I/O in synchronous code, batch API calls, parallel file processing.

**Trade-offs:** Cleaner than raw threads; GIL still limits CPU-bound parallelism — use `ProcessPoolExecutor` instead.

**Learn more:** [Python docs: concurrent.futures](https://docs.python.org/3/library/concurrent.futures.html)


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time, random

def slow_api_call(endpoint: str) -> dict:
    delay = random.uniform(0.05, 0.2)
    time.sleep(delay)
    return {"endpoint": endpoint, "status": 200, "latency_ms": int(delay * 1000)}

endpoints = [f"/api/resource/{i}" for i in range(8)]

t0 = time.perf_counter()
results = []
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(slow_api_call, ep): ep for ep in endpoints}
    for future in as_completed(futures):
        ep  = futures[future]
        res = future.result()
        print(f"  Done: {ep:22s}  latency={res['latency_ms']}ms")
        results.append(res)

print(f"\nAll {len(results)} calls completed in {(time.perf_counter()-t0)*1000:.0f}ms")
print(f"Sequential would take ~{sum(r['latency_ms'] for r in results)}ms")

# Time: O(sum(latencies) / min(workers, tasks)) wall-clock
# Space: O(workers) threads + O(tasks) futures

#### Practice — Concurrency

Pick one:

1. **Read-write lock** — implement `RWLock` that allows multiple concurrent readers but exclusive writers. Use `threading.Condition`.
2. **Async retry** — write an `async_retry(coro_fn, max_attempts, base_delay)` decorator that retries a coroutine with exponential back-off on exception.
3. **Work-stealing queue** — implement a `WorkStealingPool` where idle workers steal tasks from other workers' queues.
4. **Async pipeline** — chain three async stages (producer → transformer → sink) using `asyncio.Queue` and measure throughput vs a sequential version.

Annotate all implementations with `# Time:` and `# Space:`.

In [ ]:
# Your practice implementation here

---
# 8. Git, Testing, and Debugging

### Study checklist
- [ ] Git internals — commit graph, branching, merging
- [ ] Unit testing — assertions, edge cases, parametrisation
- [ ] Mocking — isolating the unit under test
- [ ] Structured logging and tracing
- [ ] Profiling — finding the bottleneck

### Notes
Write design decisions, gotchas, and edge cases here.

### Beginner-friendly intro
Tests catch regressions; profiling finds real bottlenecks (not imagined ones); structured logs let you debug a production incident at 2 AM. At principal level, these aren't afterthoughts — they are the job.

#### 8.1 Git Internals — Commit Graph Simulation

**Approach:** A Git repo is a DAG of commits; each commit holds a tree snapshot and parent references. Branches are just named pointers (mutable refs). Merging creates a commit with two parents.

**When to use it:** Diagnosing rebase vs merge decisions, understanding detached HEAD, explaining `git bisect`.

**Trade-offs:** Merge preserves history topology; rebase produces a linear history but rewrites SHAs.

**Learn more:** [Pro Git Book: Git Internals](https://git-scm.com/book/en/v2)


In [ ]:
import hashlib, time
from dataclasses import dataclass, field
from typing import Optional, List, Dict

@dataclass
class Commit:
    message: str
    parents: List[str]    # list of parent SHAs
    sha:     str = field(init=False)
    ts:      int = field(init=False)

    def __post_init__(self):
        self.ts  = int(time.time())
        payload  = f"{self.message}{self.parents}{self.ts}".encode()
        self.sha = hashlib.sha1(payload).hexdigest()[:8]

class GitRepo:
    def __init__(self):
        self._commits: Dict[str, Commit] = {}
        self._branches: Dict[str, str]  = {}
        self._head: str = "main"

    def commit(self, message: str) -> Commit:
        parent_sha = self._branches.get(self._head, "")
        parents    = [parent_sha] if parent_sha else []
        c = Commit(message, parents)
        self._commits[c.sha] = c
        self._branches[self._head] = c.sha
        return c

    def branch(self, name: str) -> None:
        self._branches[name] = self._branches.get(self._head, "")
        self._head = name

    def checkout(self, name: str) -> None:
        if name not in self._branches:
            raise ValueError(f"Branch {name!r} does not exist")
        self._head = name

    def merge(self, other_branch: str, message: str) -> Commit:
        current_sha = self._branches[self._head]
        other_sha   = self._branches[other_branch]
        c = Commit(message, [current_sha, other_sha])
        self._commits[c.sha] = c
        self._branches[self._head] = c.sha
        return c

    def log(self, branch: Optional[str] = None) -> None:
        branch   = branch or self._head
        sha      = self._branches.get(branch, "")
        visited  = set()
        queue    = [sha] if sha else []
        print(f"--- log: {branch} ---")
        while queue:
            s = queue.pop()
            if s in visited or s not in self._commits: continue
            visited.add(s)
            c = self._commits[s]
            print(f"  {c.sha}  {c.message}  parents={c.parents}")
            queue.extend(c.parents)

# Time: O(n) log traversal; O(1) commit/branch/checkout
# Space: O(n) commits + O(b) branches

repo = GitRepo()
repo.commit("Initial commit")
repo.commit("Add order service")
repo.branch("feature/payment")
repo.commit("Add payment stub")
repo.commit("Implement Stripe integration")
repo.checkout("main")
repo.commit("Fix logging bug")
repo.merge("feature/payment", "Merge feature/payment into main")
repo.log("main")

#### 8.2 Unit Testing — Edge Cases and Parametrisation

**Approach:** Each test covers one behaviour. Use parametrised test data for edge cases (empty, None, zero, overflow). Arrange-Act-Assert structure keeps tests readable. Name tests descriptively: `test_<function>_<scenario>`.

**When to use it:** Any production function — write tests before or immediately after.

**Trade-offs:** Tests add maintenance cost; they pay back in refactor confidence and regression detection.

**Learn more:** [Python docs: unittest](https://docs.python.org/3/library/unittest.html)


In [ ]:
import re

def parse_duration(s: str) -> int:
    # Parse strings like "2h30m", "45m", "1h", "90s" -> total seconds
    if not s: raise ValueError("Empty duration string")
    pattern = re.compile(r'(?:(\d+)h)?(?:(\d+)m)?(?:(\d+)s)?$')
    m = pattern.match(s.strip())
    if not m or not any(m.groups()):
        raise ValueError(f"Invalid duration: {s!r}")
    h, mn, sec = (int(g or 0) for g in m.groups())
    return h * 3600 + mn * 60 + sec

# --- Tests (no test runner required -- just assertions) ---
import traceback

tests = [
    # (input, expected_output_or_exception)
    ("1h",        3600),
    ("30m",       1800),
    ("2h30m",     9000),
    ("90s",         90),
    ("1h2m3s",    3723),
    ("",          ValueError),
    ("abc",       ValueError),
    ("0m",           0),
    (" 5m ",       300),   # whitespace tolerance
]

passed = failed = 0
for inp, expected in tests:
    try:
        result = parse_duration(inp)
        if isinstance(expected, type) and issubclass(expected, Exception):
            print(f"FAIL  parse_duration({inp!r}): expected {expected.__name__}, got {result}")
            failed += 1
        elif result == expected:
            print(f"PASS  parse_duration({inp!r}) == {result}")
            passed += 1
        else:
            print(f"FAIL  parse_duration({inp!r}): expected {expected}, got {result}")
            failed += 1
    except Exception as e:
        if isinstance(expected, type) and isinstance(e, expected):
            print(f"PASS  parse_duration({inp!r}) raises {type(e).__name__}")
            passed += 1
        else:
            print(f"FAIL  parse_duration({inp!r}): unexpected {type(e).__name__}: {e}")
            failed += 1

print(f"\n{passed} passed, {failed} failed")

#### 8.3 Mocking — Isolating the Unit Under Test

**Approach:** Replace external dependencies (email, DB, HTTP) with `unittest.mock.MagicMock`. Verify the unit's behaviour without side effects. Use `assert_called_once_with` to check call contracts.

**When to use it:** Any code that calls external services, the filesystem, or the clock.

**Trade-offs:** Mocks test behaviour, not integration. Always complement with integration tests against real services.

**Learn more:** [Python docs: unittest.mock](https://docs.python.org/3/library/unittest.mock.html)


In [ ]:
from unittest.mock import MagicMock, patch, call

class UserRepo:
    def find_by_id(self, uid: int): pass   # real: DB call

class EmailService:
    def send(self, to: str, subject: str, body: str): pass  # real: SMTP

class WelcomeWorkflow:
    def __init__(self, repo: UserRepo, email: EmailService):
        self._repo  = repo
        self._email = email

    def run(self, user_id: int) -> bool:
        user = self._repo.find_by_id(user_id)
        if user is None:
            return False
        self._email.send(user["email"],
                         "Welcome!",
                         f"Hi {user['name']}, welcome aboard.")
        return True

# --- Unit test with mocks ---
mock_repo  = MagicMock(spec=UserRepo)
mock_email = MagicMock(spec=EmailService)

mock_repo.find_by_id.return_value = {"id": 42, "name": "Alice", "email": "alice@example.com"}

wf = WelcomeWorkflow(mock_repo, mock_email)
result = wf.run(42)

assert result is True
mock_repo.find_by_id.assert_called_once_with(42)
mock_email.send.assert_called_once_with(
    "alice@example.com", "Welcome!", "Hi Alice, welcome aboard."
)
print("All assertions passed")

# Test the 'user not found' path
mock_repo.find_by_id.return_value = None
result = wf.run(99)
assert result is False
print("Not-found path: correct")

#### 8.4 Structured Logging and Tracing

**Approach:** Emit JSON log lines with consistent fields (`timestamp`, `level`, `service`, `trace_id`, `msg`). A `@trace` decorator injects a `trace_id` into every log call within a request, making distributed debugging tractable.

**When to use it:** Any production service. Structure enables log aggregation tools (Datadog, Splunk, CloudWatch Logs Insights).

**Trade-offs:** JSON logs are larger and slightly slower to emit; the debuggability payoff is enormous in production.

**Learn more:** [Python docs: logging](https://docs.python.org/3/library/logging.html)


In [ ]:
import json, time, uuid, logging, functools
from typing import Callable

class StructuredLogger:
    def __init__(self, service: str):
        self.service   = service
        self._trace_id = None

    def set_trace(self, trace_id: str): self._trace_id = trace_id

    def _emit(self, level: str, msg: str, **extra):
        record = {
            "ts":       time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "level":    level,
            "service":  self.service,
            "trace_id": self._trace_id,
            "msg":      msg,
            **extra,
        }
        print(json.dumps(record))

    def info(self,  msg, **kw): self._emit("INFO",  msg, **kw)
    def warn(self,  msg, **kw): self._emit("WARN",  msg, **kw)
    def error(self, msg, **kw): self._emit("ERROR", msg, **kw)

log = StructuredLogger("order-service")

def trace(logger: StructuredLogger):
    def decorator(fn: Callable) -> Callable:
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            tid = str(uuid.uuid4())[:8]
            logger.set_trace(tid)
            logger.info(f"START {fn.__name__}")
            t0 = time.perf_counter()
            try:
                result = fn(*args, **kwargs)
                logger.info(f"END {fn.__name__}", latency_ms=round((time.perf_counter()-t0)*1000))
                return result
            except Exception as e:
                logger.error(f"FAIL {fn.__name__}", error=str(e))
                raise
        return wrapper
    return decorator

# Time: O(1) per log call; O(f) per traced function where f = function runtime
# Space: O(1)

@trace(log)
def process_order(order_id: str, amount: float) -> dict:
    log.info("Validating order", order_id=order_id)
    if amount <= 0:
        raise ValueError("amount must be positive")
    log.info("Order accepted", order_id=order_id, amount=amount)
    return {"order_id": order_id, "status": "accepted"}

process_order("ORD-42", 99.99)

#### 8.5 Profiling — Find the Real Bottleneck

**Approach:** `cProfile` records cumulative call time per function. Visualise with `pstats` or `snakeviz`. Always profile before optimising — the bottleneck is rarely where you think it is.

**When to use it:** Before any performance optimisation. Also for regression testing: baseline → change → compare.

**Trade-offs:** `cProfile` adds ~10-50% overhead; use `perf` sampling profiler for minimal overhead in production.

**Learn more:** [Python docs: profile and cProfile](https://docs.python.org/3/library/profile.html)


In [ ]:
import cProfile, pstats, io

def fib_naive(n: int) -> int:
    # Time: O(2^n)  Space: O(n) call stack
    if n < 2: return n
    return fib_naive(n-1) + fib_naive(n-2)

def fib_memo(n: int, memo=None) -> int:
    # Time: O(n)  Space: O(n)
    if memo is None: memo = {}
    if n in memo: return memo[n]
    if n < 2: return n
    memo[n] = fib_memo(n-1, memo) + fib_memo(n-2, memo)
    return memo[n]

def profile(fn, *args, top_n=5):
    pr = cProfile.Profile()
    pr.enable()
    result = fn(*args)
    pr.disable()

    buf = io.StringIO()
    ps  = pstats.Stats(pr, stream=buf).sort_stats('cumulative')
    ps.print_stats(top_n)
    print(f"--- {fn.__name__}({', '.join(map(str,args))}) = {result} ---")
    print(buf.getvalue())
    return result

profile(fib_naive, 28)
profile(fib_memo,  500)

#### Practice — Git, Testing, Debugging

Pick one:

1. **Git rebase simulation** — extend `GitRepo` with a `rebase(source_branch, onto_branch)` method that replays source commits on top of the onto branch tip, producing new SHAs.
2. **Property-based testing** — use `hypothesis` (or write your own) to generate random duration strings and verify that `parse_duration(fmt(seconds)) == seconds` holds for all valid inputs.
3. **Log sampling** — extend `StructuredLogger` with a `sample_rate` parameter: only emit `INFO` logs with probability `sample_rate` to reduce volume in high-traffic paths.
4. **Line profiler** — instrument `fib_naive` line-by-line using `sys.settrace` (a simplified version of `line_profiler`) and print hit counts per line.

Annotate all implementations with `# Time:` and `# Space:`.

In [ ]:
# Your practice implementation here

---
# Review — Principal SWE Coverage Map

| Area | Key Concepts | Interview Signal |
|------|-------------|------------------|
| **OOP** | Encapsulation, SOLID, ABCs, dunders | Design a class hierarchy; spot the fragile-base-class problem |
| **Design Patterns** | Singleton, Factory, Builder, Observer, Strategy, Decorator, Adapter | 'How would you extend this without modifying it?' |
| **System Design** | Load balancing, LRU, rate limiting, CAP, consistent hashing | 'Design a distributed cache / rate limiter / URL shortener' |
| **OS Basics** | GIL, stack vs heap, deadlock, file descriptors, scheduling | 'Why does your CPU-bound Python not scale with threads?' |
| **DBMS** | ACID, B-tree, JOINs, window functions, normalisation | 'Schema critique; optimise this slow query; explain ACID' |
| **Networking** | TCP, HTTP, DNS, REST, TLS | 'Debug a latency spike; design a REST API; explain TLS' |
| **Concurrency** | Race conditions, semaphores, producer-consumer, asyncio | 'Implement a thread-safe pool; when do you use async vs threads?' |
| **Git/Test/Debug** | Commit DAG, unit tests, mocking, structured logs, profiling | 'How do you test untestable code? Find the bottleneck?' |

---

**Next steps after completing this notebook:**

- Pair each pattern with a real ticket or PR at work
- Run `git log --graph --oneline` on a large repo and trace the DAG
- Profile a slow endpoint in your own service using `cProfile` or `py-spy`
- Write one integration test that replaces a mock with a real dependency
